# Design of Computer Programs — Final Exam

Solutions and notes for the final exam of Udacity's *Design of Computer Programs* (CS212, Peter Norvig).

## Unit 1 — Bowling

**Problem.** Write `bowling(balls)`, which returns the integer score of a ten-pin bowling game. `balls` is a list of integers, each one the number of pins knocked down by a ball. A perfect game scores `300`.

**Rules (summary).**
- A game has **10 frames**. You roll 1 or 2 balls per frame, except the 10th frame, where you roll 2 or 3.
- **Open frame** (fewer than 10 pins with two balls): score the pins knocked down.
- **Spare** (all 10 pins on the *second* ball): score `10` plus the *next* ball as a bonus.
- **Strike** (all 10 pins on the *first* ball): score `10` plus the *next two* balls as a bonus.
- Bonus balls also count toward their own frames, except in the 10th frame.

**Key idea.** Walk through exactly 10 frames. For each frame decide how many balls it *uses up* and how many balls *count toward its score*:

| Frame type | Balls used | Balls scored |
|------------|------------|--------------|
| Strike     | 1          | 3 (the strike + next two) |
| Spare      | 2          | 3 (the two balls + next one) |
| Open       | 2          | 2 |

Summing the scored balls per frame naturally handles the strike/spare bonuses, because the bonus balls are simply re-counted by the current frame while still being "used up" by later frames.


In [1]:
def bowling(balls):
    "Compute the total score for a player's game of bowling."
    ## bowling([int, ...]) -> int
    total = 0
    for frame in range(10):
        score, balls = score_frame(balls)
        total += score
    return total


def score_frame(balls):
    "Return two values: (score_for_this_frame, remaining_balls)."
    n_used, n_scoring = ((1, 3) if balls[0] == 10            # strike
                    else (2, 3) if balls[0] + balls[1] == 10  # spare
                    else (2, 2))                              # open frame
    return (sum(balls[:n_scoring]), balls[n_used:])


In [2]:
def test_bowling():
    assert   0 == bowling([0] * 20)
    assert  20 == bowling([1] * 20)
    assert  80 == bowling([4] * 20)
    assert 190 == bowling([9,1] * 10 + [9])
    assert 300 == bowling([10] * 12)
    assert 200 == bowling([10, 5,5] * 5 + [10])
    assert  11 == bowling([0,0] * 9 + [10,1,0])
    assert  12 == bowling([0,0] * 8 + [10, 1,0])
    assert  20 == bowling([0,0] * 9 + [10, 10,0])
    assert  30 == bowling([0,0] * 8 + [10, 10,0])
    assert 168 == bowling([9,1, 0,10, 10, 10, 6,2, 7,3, 8,2, 10, 9,0, 9,1,10])
    assert  24 == bowling([10, 3, 4] + [0] * 17)
    assert 168 == bowling([10, 7,3, 7,2, 9,1, 10, 10, 10, 2,3, 6,4, 7,3,3])
    assert 133 == bowling([1,4, 4,5, 6,4, 5,5, 10, 0,1, 7,3, 6,4, 10, 2,8,6])
    assert  16 == bowling([5,5, 3,0] + [0,0] * 8)
    assert 200 == bowling([5,5] + [10, 5,5] * 5)
    assert  20 == bowling([0,0] * 9 + [10,5,5])
    return 'tests pass'

print(test_bowling())


tests pass


### Alternative solution (mutating `balls`)

`score_frame` does most of the work but hides the "sum of 10 frames" logic behind a 2-value return. An alternative makes the summation explicit by **mutating** `balls` instead of returning the remaining list:


In [3]:
def bowling1(balls):
    "Compute the score for one player's game of bowling."
    return sum(score_frame1(balls) for frame in range(10))


def score_frame1(balls):
    "Return the score for this frame and remove its used balls from `balls`."
    n_used, n_scoring = ((1, 3) if balls[0] == 10            # strike
                    else (2, 3) if balls[0] + balls[1] == 10  # spare
                    else (2, 2))                              # open frame
    score = sum(balls[:n_scoring])
    balls[:n_used] = []
    return score


# bowling1 mutates its input, so pass a fresh copy each time it's tested.
assert 300 == bowling1([10] * 12)
assert 190 == bowling1([9, 1] * 10 + [9])
print('alternative solution: tests pass')


alternative solution: tests pass


## Unit 2 — Logic Puzzle

**Problem.** Five people (Wilkes, Hamming, Minsky, Knuth, Simon) arrive Monday–Friday. Each has a profession (programmer, writer, manager, designer, plus one unnamed) and bought a device (laptop, droid, tablet, iphone, plus one unnamed). Twelve clues constrain who arrived when. Write `logic_puzzle()` returning the names in arrival order.

**Key idea — represent everything as a day number.** Days are `Mon=1 … Fri=5`. Every attribute (a person, a profession, a device) is assigned *the day on which it occurs*. Two attributes that share the same value happened on the same day. This turns the puzzle into a constraint search over permutations of `(1, 2, 3, 4, 5)`:

- One permutation assigns a day to each **person**.
- Another assigns a day to each **profession**.
- Another assigns a day to each **device**.

We brute-force the permutations and keep only the assignment satisfying all 12 clues. To make it fast, clues are tested **as early as possible** (right after the permutation they depend on is chosen), pruning the search tree. Finally `answer` turns the `{name: day}` mapping into a list sorted by day.

**Solution:** `['Wilkes', 'Simon', 'Knuth', 'Hamming', 'Minsky']`.


In [4]:
import itertools


def logic_puzzle():
    "Return a list of the names of the people, in the order they arrive."
    days = (mon, tue, wed, thu, fri) = (1, 2, 3, 4, 5)
    possible_days = list(itertools.permutations(days))
    return next(answer(Wilkes=Wilkes, Hamming=Hamming, Minsky=Minsky,
                       Knuth=Knuth, Simon=Simon)
                for (Wilkes, Hamming, Minsky, Knuth, Simon) in possible_days
                if Knuth == Simon + 1                                # 6
                for (programmer, writer, manager, designer, _) in possible_days
                if Knuth == manager + 1                              # 10
                and thu != designer                                 # 7
                and programmer != Wilkes and writer != Minsky       # 2, 4
                for (laptop, droid, tablet, iphone, _) in possible_days
                if set([laptop, Wilkes]) == set([mon, writer])       # 11
                and set([programmer, droid]) == set([Wilkes, Hamming])  # 3
                and (iphone == tue or tablet == tue)                 # 12
                and designer != droid                               # 9
                and Knuth != manager and tablet != manager          # 5
                and wed == laptop                                   # 1
                and fri != tablet                                  # 8
                )


def answer(**names):
    "Given a dict of {name: day}, return a list of names sorted by day."
    return sorted(names, key=lambda name: names[name])


assert logic_puzzle() == ['Wilkes', 'Simon', 'Knuth', 'Hamming', 'Minsky']
print(logic_puzzle())


['Wilkes', 'Simon', 'Knuth', 'Hamming', 'Minsky']


## Unit 3 — Functions and APIs: Polynomials

**Problem.** Represent a polynomial such as `30 * x**2 + 20 * x + 10` as a **callable function** built from its coefficient tuple `(10, 20, 30)` (coefficient of `x**n` sits at index `n`). The function also carries:
- `.coefs` — the coefficient tuple, and
- `.__name__` — the simplified human-readable formula (drop `0 * x**n`, simplify `1 * x**n` → `x**n`, `x**1` → `x`, `c * x**0` → `c`).

Then implement the API: `is_poly`, `add`, `sub`, `mul`, `power`, `deriv`, `integral`, plus extra credit (`poly` as a class with overloaded operators, and `Poly` parsing a string).

**Key ideas.**
- **Compile once, call fast.** Instead of re-summing terms on every call, build the evaluation function with a single `eval('lambda x: ...')`, using **Horner's rule** (`(c0 + x*(c1 + x*(c2 + ...)))`) to minimise multiplications.
- **Memoization.** `poly` delegates to a memoized `polynomial`, so equal coefficients always yield the *same* function object.
- **Canonical coefs.** Strip trailing zeros and coerce scalars to tuples so equal polynomials compare equal.
- **Calculus is term-wise.** Derivative of `c * x**n` is `c*n * x**(n-1)`; integral is `c/(n+1) * x**(n+1)` (plus constant `C`).
- **Extra credit.** Defining `poly` as a class with `__add__/__sub__/__mul__/__pow__` (and the `__r*__` reflected versions) lets you write polynomials naturally as `30 * x**2 + 20 * x + 10`; `coerce_poly` promotes plain numbers to polynomials.


In [5]:
from collections import defaultdict
import functools


def decorator(d):
    "Make function d a decorator: d wraps a function fn."
    def _d(fn):
        return functools.update_wrapper(d(fn), fn)
    functools.update_wrapper(_d, d)
    return _d


@decorator
def memo(f):
    """Decorator that caches the return value for each call to f(*args).
    Then when called again with same args, we can just look it up."""
    cache = {}
    def _f(*args):
        try:
            return cache[args]
        except KeyError:
            result = f(*args)
            try:
                cache[args] = result
            except TypeError:  # args refuses to be a dict key
                pass
            return result
    _f.cache = cache
    return _f


def same_name(name1, name2):
    """Use same_name rather than name1 == name2 to allow for some
    variation in naming conventions."""
    def canonical_name(name):
        return name.replace(' ', '').replace('+-', '-')
    return canonical_name(name1) == canonical_name(name2)


In [6]:
def poly(coefs):
    """Return a function that is the polynomial with these coefficients.
    For example if coefs=(10, 20, 30) return the function of x that computes
    '30 * x**2 + 20 * x + 10'.  Also store coefs on the .coefs attribute of
    the function, and the str of the formula on the .__name__ attribute."""
    return polynomial(canonical(coefs))


@memo
def polynomial(coefs):
    """Return a polynomial function with these attributes.  Memoized, so any
    two polys with the same coefficients will be identical polys."""
    p = eval('lambda x: ' + horner_formula(coefs), {})
    p.__name__ = polynomial_formula(coefs)
    p.coefs = coefs
    return p


def horner_formula(coefs):
    """A relatively efficient form to evaluate a polynomial.
    E.g.: horner_formula((10, 20, 30, 0, -50))
          == '(10 + x * (20 + x * (30 + x * x * -50)))',
    which is 4 multiplies and 3 adds."""
    c = coefs[0]
    if len(coefs) == 1:
        return str(c)
    else:
        factor = 'x * ' + horner_formula(coefs[1:])
        return factor if c == 0 else '(%s + %s)' % (c, factor)


def polynomial_formula(coefs):
    """A simple human-readable form for a polynomial.
    E.g.: polynomial_formula((10, 20, 30, 0, -50))
          == '-50 * x**4 + 30 * x**2 + 20 * x + 10'."""
    terms = [term(c, n)
             for (n, c) in reversed(list(enumerate(coefs))) if c != 0]
    return ' + '.join(terms)


def term(c, n):
    "Return a string representing 'c * x**n' in simplified form."
    if n == 0:
        return str(c)
    xn = 'x' if (n == 1) else ('x**' + str(n))
    return xn if (c == 1) else '-' + xn if (c == -1) else str(c) + ' * ' + xn


def canonical(coefs):
    "Canonicalize coefs by dropping trailing zeros and converting to a tuple."
    if not coefs:
        coefs = [0]
    elif isinstance(coefs, (int, float)):
        coefs = [coefs]
    else:
        coefs = list(coefs)
    while coefs[-1] == 0 and len(coefs) > 1:
        del coefs[-1]
    return tuple(coefs)


In [7]:
def is_poly(x):
    "Return true if x is a poly (polynomial)."
    return callable(x) and hasattr(x, 'coefs')


def add(p1, p2):
    "Return a new polynomial which is the sum of polynomials p1 and p2."
    N = max(len(p1.coefs), len(p2.coefs))
    coefs = [0] * N
    for (n, c) in enumerate(p1.coefs):
        coefs[n] = c
    for (n, c) in enumerate(p2.coefs):
        coefs[n] += c
    return poly(coefs)


def sub(p1, p2):
    "Return a new polynomial which is p1 - p2."
    N = max(len(p1.coefs), len(p2.coefs))
    coefs = [0] * N
    for (n, c) in enumerate(p1.coefs):
        coefs[n] = c
    for (n, c) in enumerate(p2.coefs):
        coefs[n] -= c
    return poly(coefs)


def mul(p1, p2):
    "Return a new polynomial which is the product of polynomials p1 and p2."
    # Given terms a*x**n and b*x**m, accumulate a*b in results[n+m]
    results = defaultdict(int)
    for (n, a) in enumerate(p1.coefs):
        for (m, b) in enumerate(p2.coefs):
            results[n + m] += a * b
    return poly([results[i] for i in range(max(results) + 1)])


def power(p, n):
    "Return a poly which is p to the nth power (n a non-negative integer)."
    if n == 0:
        return poly((1,))
    elif n == 1:
        return p
    elif n % 2 == 0:
        return square(power(p, n // 2))
    else:
        return mul(p, power(p, n - 1))


def square(p):
    return mul(p, p)


def deriv(p):
    "Return the derivative of a function p (with respect to its argument)."
    return poly([n * c for (n, c) in enumerate(p.coefs) if n > 0])


def integral(p, C=0):
    "Return the integral of a function p (with respect to its argument)."
    return poly([C] + [float(c) / (n + 1) for (n, c) in enumerate(p.coefs)])


In [8]:
def test_poly():
    global p1, p2, p3, p4, p5, p9  # global to ease debugging in an interactive session

    p1 = poly((10, 20, 30))
    assert p1(0) == 10
    for x in (1, 2, 3, 4, 5, 1234.5):
        assert p1(x) == 30 * x**2 + 20 * x + 10
    assert same_name(p1.__name__, '30 * x**2 + 20 * x + 10')

    assert is_poly(p1)
    assert not is_poly(abs) and not is_poly(42) and not is_poly('cracker')

    p3 = poly((0, 0, 0, 1))
    assert p3.__name__ == 'x**3'
    p9 = mul(p3, mul(p3, p3))
    assert p9 == poly([0, 0, 0, 0, 0, 0, 0, 0, 0, 1])
    assert p9(2) == 512
    p4 = add(p1, p3)
    assert same_name(p4.__name__, 'x**3 + 30 * x**2 + 20 * x + 10')

    assert same_name(poly((1, 1)).__name__, 'x + 1')
    assert same_name(power(poly((1, 1)), 10).__name__,
        'x**10 + 10 * x**9 + 45 * x**8 + 120 * x**7 + 210 * x**6 + 252 * x**5 + 210' +
        ' * x**4 + 120 * x**3 + 45 * x**2 + 10 * x + 1')

    assert add(poly((10, 20, 30)), poly((1, 2, 3))) == poly((11, 22, 33))
    assert sub(poly((10, 20, 30)), poly((1, 2, 3))) == poly((9, 18, 27))
    assert mul(poly((10, 20, 30)), poly((1, 2, 3))) == poly((10, 40, 100, 120, 90))
    assert power(poly((1, 1)), 2) == poly((1, 2, 1))
    assert power(poly((1, 1)), 10) == poly((1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1))

    assert deriv(p1) == poly((20, 60))
    assert integral(poly((20, 60))) == poly((0, 20, 30))
    p5 = poly((0, 1, 2, 3, 4, 5))
    assert same_name(p5.__name__, '5 * x**5 + 4 * x**4 + 3 * x**3 + 2 * x**2 + x')
    assert p5(1) == 15
    assert p5(2) == 258
    assert same_name(deriv(p5).__name__, '25 * x**4 + 16 * x**3 + 9 * x**2 + 4 * x + 1')
    assert deriv(p5)(1) == 55
    assert deriv(p5)(2) == 573
    # Additional test case:
    p6 = poly((1,))
    assert integral(p6)(10) == 10
    return 'test_poly passes'


print(test_poly())


test_poly passes


### Extra credit — `poly` as a class and `Poly` from a string

Redefining `poly` as a **class** with overloaded operators (`__add__`, `__sub__`, `__mul__`, `__pow__`, plus the reflected `__radd__`/`__rmul__`) lets us build polynomials with natural arithmetic, e.g. `30 * x**2 + 20 * x + 10`. `coerce_poly` promotes plain numbers so `x + 1` and `1 + x` both work. `Poly('...')` then just `eval`s the formula in an environment where `x` is the polynomial `poly((0, 1))`.

> Note: this cell **redefines** `poly` and `is_poly` as the class-based versions, replacing the function-based ones above.


In [9]:
class poly(object):
    """poly objects are like the poly functions defined earlier, but are
    objects of a class. We coerce arguments to poly, so you can do (x + 1)
    and the 1 will be converted to a poly first."""

    def __init__(self, coefs):
        coefs = canonical(coefs)
        self.fn = eval('lambda x: ' + horner_formula(coefs), {})
        self.__name__ = polynomial_formula(coefs)
        self.coefs = coefs

    def __call__(self, x):
        return self.fn(x)

    def __eq__(self, other):
        return isinstance(other, poly) and self.coefs == other.coefs

    def __add__(self, p2):
        return add(self, coerce_poly(p2))   # p + p2

    def __sub__(self, p2):
        return sub(self, coerce_poly(p2))   # p - p2

    def __mul__(self, p2):
        return mul(self, coerce_poly(p2))   # p * p2

    def __pow__(self, n):
        return power(self, n)               # p ** n

    def __neg__(self):
        return poly([-c for c in self.coefs])  # - p

    def __pos__(self):
        return self                          # + p

    # The reflected methods make 1 + x work as well as x + 1.
    def __rmul__(self, p2):
        return mul(self, coerce_poly(p2))    # 5 * x

    def __radd__(self, p2):
        return add(self, coerce_poly(p2))    # 1 + x

    def __hash__(self):
        return hash(self.coefs)

    def __repr__(self):
        return '<%s>' % self.__name__


def coerce_poly(p):
    "Make this into a poly if it isn't already."
    return p if isinstance(p, poly) else poly(p)


def is_poly(p):
    return isinstance(p, poly)


def Poly(formula):
    "Parse the formula using eval in an environment where x is a poly."
    return eval(formula, {'x': poly((0, 1))})


In [10]:
def test_poly1():
    # Define x as the polynomial 1*x + 0.
    x = poly((0, 1))
    # From here on we can create polynomials by + and * operations on x.
    newp1 = 30 * x**2 + 20 * x + 10  # This is a poly object, not a number!
    assert p1(100) == newp1(100)     # The new poly objects are still callable.
    assert same_name(p1.__name__, newp1.__name__)
    assert (x + 1) * (x - 1) == x**2 - 1 == poly((-1, 0, 1))
    return 'test_poly1 passes'


def test_poly2():
    newp1 = Poly('30 * x**2 + 20 * x + 10')
    assert p1(100) == newp1(100)
    assert same_name(p1.__name__, newp1.__name__)
    return 'test_poly2 passes'


print(test_poly1())
print(test_poly2())


test_poly1 passes
test_poly2 passes


## Unit 4 — Search (Parking Lot Puzzle)

**Problem.** Slide cars in a parking lot (a Rush-Hour-style puzzle) until the `*` car overlaps the goal `@`. Cars move only along their orientation, any number of free squares. Write `solve_parking_puzzle(start, N=8)` returning an alternating `[state, action, state, action, ...]` path, where an action is `(car, distance)` and distance is a multiple of `±1` (horizontal) or `±N` (vertical).

**Key ideas.**
- **Grid as flat indices.** The `N×N` board is a single list of `N**2` cells. Horizontal neighbours differ by `1`, vertical neighbours by `N`. A car is `(name, locations)`; the whole state is a tuple of such pairs (including the walls `'|'` and goal `'@'`).
- **Reuse generic search.** With a `successors` function and an `is_goal` test, the unit's `shortest_path_search` (BFS) finds the shortest solution — no puzzle-specific search code needed.
- **`is_goal`.** The `*` car's squares intersect the `@` square.
- **`psuccessors`.** For every movable car, slide it from `max(locs)` forward and from `min(locs)` backward, one step at a time, by `±diff` (where `diff = locs[1]-locs[0]` is `1` or `N`), stopping at the first occupied square. Each reachable position is a successor keyed by its action `(car, distance)`.
- **Canonical states.** `update` rebuilds the state via a dict and sorts the items, so equal positions are the same tuple (good for the `explored` set).
- **Helpers.** `locs(start, n, incr)` builds a car's squares; `grid(cars, N)` adds the surrounding walls and the goal in the middle of the right wall; `show` prints a state.

**Solution lengths:** `puzzle1` → 4 moves, `puzzle2` → 7, `puzzle3` → 7, `puzzle4` → 8.


In [11]:
# Generic search helpers from the unit, plus locs / grid / show.

def shortest_path_search(start, successors, is_goal):
    """Find the shortest path from start state to a state
    such that is_goal(state) is true."""
    if is_goal(start):
        return [start]
    explored = set()        # set of states we have visited
    frontier = [[start]]    # ordered list of paths we have blazed
    while frontier:
        path = frontier.pop(0)
        s = path[-1]
        for (state, action) in successors(s).items():
            if state not in explored:
                explored.add(state)
                path2 = path + [action, state]
                if is_goal(state):
                    return path2
                else:
                    frontier.append(path2)
    return []


def path_actions(path):
    "Return a list of actions in this path."
    return path[1::2]


N = 8


def locs(start, n, incr=1):
    "Return a tuple of n locations, starting at start and go up by incr."
    return tuple(start + i * incr for i in range(n))


def grid(cars, N=N):
    """Return a tuple of (object, locations) pairs -- the format expected for
    this puzzle, adding walls all around the NxN grid (except the goal in the
    middle of the right-hand wall) and the goal pair like ('@', (31,))."""
    goals = ((N**2) // 2 - 1,)
    walls = (locs(0, N) + locs(N * (N - 1), N) + locs(N, N - 2, N)
             + locs(2 * N - 1, N - 2, N))
    walls = tuple(w for w in walls if w not in goals)
    return cars + (('|', walls), ('@', goals))


def show(state, N=N):
    "Print a representation of a state as an NxN grid."
    board = ['.'] * N**2
    for (c, squares) in state:
        for s in squares:
            board[s] = c
    for i, s in enumerate(board):
        print(s, end=' ')
        if i % N == N - 1:
            print()


In [12]:
def solve_parking_puzzle(start, N=N):
    """Solve the puzzle described by the starting position (a tuple
    of (object, locations) pairs).  Return a path of [state, action, ...]
    alternating items; an action is a pair (object, distance_moved),
    such as ('B', 16) to move 'B' two squares down on the N=8 grid."""
    return shortest_path_search(start, psuccessors, is_goal)


def is_goal(state):
    "Goal is when the car (*) overlaps a goal square (@)."
    d = dict(state)
    return set(d['*']) & set(d['@'])


def psuccessors(state):
    """State is a tuple of (('c', sqs), ...); return a {state: action} dict
    where action is of form ('c', dir), where dir is +/-1 or +/-N."""
    results = {}
    occupied = set(s for (c, sqs) in state for s in sqs if c != '@')
    for (c, sqs) in state:
        if c not in '|@':  # Walls and goals can't move
            diff = sqs[1] - sqs[0]
            # Either move the max of sqs up, or the min of sqs down
            for (d, start) in [(diff, max(sqs)), (-diff, min(sqs))]:
                for i in range(1, N - 2):
                    s = start + d * i
                    if s in occupied:
                        break  # Stop when you hit something
                    results[update(state, c, tuple(q + d * i for q in sqs))] = (c, d * i)
    return results


def update(tuples, key, val):
    "Return a new tuple of pairs, dropping old value of key and adding new."
    # Sort the keys to make sure the result is canonical.
    d = dict(tuples)
    d[key] = val
    return tuple(sorted(d.items()))


In [13]:
puzzle1 = grid((
    ('*', locs(26, 2)),
    ('G', locs(9, 2)),
    ('Y', locs(14, 3, N)),
    ('P', locs(17, 3, N)),
    ('O', locs(41, 2, N)),
    ('B', locs(20, 3, N)),
    ('A', locs(45, 2))))

puzzle2 = grid((
    ('*', locs(26, 2)),
    ('B', locs(20, 3, N)),
    ('P', locs(33, 3)),
    ('O', locs(41, 2, N)),
    ('Y', locs(51, 3))))

puzzle3 = grid((
    ('*', locs(25, 2)),
    ('B', locs(19, 3, N)),
    ('P', locs(36, 3)),
    ('O', locs(45, 2, N)),
    ('Y', locs(49, 3))))

puzzle4 = grid((
    ('*', locs(26, 2)),
    ('G', locs(9, 2)),
    ('Y', locs(14, 3, N)),
    ('P', locs(17, 3, N)),
    ('O', locs(41, 2, N)),
    ('B', locs(20, 3, N)),
    ('A', locs(45, 2)),
    ('S', locs(51, 3))))


def same_state(state1, state2):
    "Two states are the same if all corresponding sets of locs are the same."
    d1, d2 = dict(state1), dict(state2)
    return all(set(d1[key]) == set(d2[key]) for key in set(d1) | set(d2))


def legal_step(path):
    "A legal step has an action that leads to a valid successor state."
    state1, action, state2 = path
    succs = psuccessors(state1)
    return state2 in succs and succs[state2] == action


def valid_solution(puzzle, length):
    "Does solve_parking_puzzle solve this puzzle in length steps?"
    path = solve_parking_puzzle(puzzle)
    return (len(path_actions(path)) == length and
            same_state(path[0], puzzle) and
            is_goal(path[-1]) and
            all(legal_step(path[i:i + 3]) for i in range(0, len(path) - 2, 2)))


def test_parking():
    assert valid_solution(puzzle1, 4)
    assert valid_solution(puzzle2, 7)
    assert valid_solution(puzzle3, 7)
    assert valid_solution(puzzle4, 8)
    assert locs(26, 2) == (26, 27)
    assert locs(20, 3, 8) == (20, 28, 36)
    return 'test_parking passes'


print(test_parking())
print()
print('puzzle1 solution:', path_actions(solve_parking_puzzle(puzzle1)))


test_parking passes

puzzle1 solution: [('A', -3), ('B', 16), ('Y', 24), ('*', 4)]


## Unit 5 — Probability in the game of Darts

**Part A — `double_out(total)`.** Return a shortest list of 1–3 dart targets that sum to `total`, where the **last** dart is a double (the "double-out" rule of 501). Return `None` if impossible.

**Key ideas.**
- Precompute every score a single dart can make (`points`) and the set of valid `doubles`.
- Build `ordered_points = [0] + sorted(points, reverse=True)`. Including **0** lets a single nested loop cover the 1-, 2-, and 3-dart cases at once (a 0 is just "no dart"), and the descending order makes us prefer high-value targets first.
- Loop `dart1, dart2` over `ordered_points`; `dart3` is whatever is left. If `dart3` is a valid double, we have the shortest solution. `name(d)` converts a numeric score into a target name, preferring the easiest ring (`S`, then `T`, then `D`) — but the final dart must be a `D`.

**Part B — `outcome(target, miss)` and `best_target(miss)`.** Model an inaccurate thrower with a single `miss` ratio. Misses are **independent** in two dimensions, so `P(ring, section) = P(ring) * P(section)`.

**Key ideas.**
- `ring_outcome` distributes ring misses per the rules (thin triple → single; double → half single / half `OFF`; thick single → 1/5 miss split to double & triple; bull rules for `SB`/`DB`).
- `section_outcome` sends half the section misses to each clockwise/counter-clockwise neighbour (or, for a bull, equally to all 20 sections).
- `outcome` combines them by multiplying probabilities, with the special case: aiming at a bull but missing on the *ring* lands in some single section, spread equally over all 20.
- `best_target(miss)` maximises `expected_value` = `sum(value(t) * p)` over the outcome distribution. As `miss` grows the optimum drifts from `T20` toward lower, safer sections (`T19`, `T7`, ...).


In [14]:
# Part A: double_out

singles = list(range(1, 21)) + [25]
points = set(m * s for s in singles for m in (1, 2, 3) if m * s != 75)
doubles = set(2 * s for s in singles)
ordered_points = [0] + sorted(points, reverse=True)


def double_out(total):
    """Return a shortest possible list of targets that add to total,
    where the length <= 3 and the final element is a double.
    If there is no solution, return None."""
    if total > 60 + 60 + 50:
        return None
    for dart1 in ordered_points:
        for dart2 in ordered_points:
            dart3 = total - dart1 - dart2
            if dart3 in doubles:
                solution = [name(dart1), name(dart2), name(dart3, 'D')]
                return [t for t in solution if t != 'OFF']
    return None


def name(d, double=False):
    """Given an int, d, return the name of a target that scores d.
    If double is true, the name must start with 'D', otherwise,
    prefer the order 'S', then 'T', then 'D'."""
    return ('OFF' if d == 0 else
            'DB' if d == 50 else
            'SB' if d == 25 else
            'D' + str(d // 2) if (d in doubles and double) else
            'S' + str(d) if d in singles else
            'T' + str(d // 3) if (d % 3 == 0) else
            'D' + str(d // 2))


def value(target):
    "The numeric value of a target."
    if target == 'OFF':
        return 0
    ring, section = target[0], target[1:]
    r = 'OSDT'.index(ring)
    s = 25 if section == 'B' else int(section)
    return r * s


def valid_out(darts, total):
    "Does this list of targets achieve the total, and end with a double?"
    return (0 < len(darts) <= 3 and darts[-1].startswith('D')
            and sum(map(value, darts)) == total)


def test_darts():
    "Test the double_out function."
    assert double_out(170) == ['T20', 'T20', 'DB']
    assert double_out(171) is None
    assert double_out(100) in (['T20', 'D20'], ['DB', 'DB'])
    for total in list(range(2, 159)) + [160, 161, 164, 167, 170]:
        assert valid_out(double_out(total), total)
    for total in [0, 1, 159, 162, 163, 165, 166, 168, 169, 171, 200]:
        assert double_out(total) is None
    return 'test_darts passes'


print(test_darts())


test_darts passes


In [15]:
# Part B: outcome and best_target

from collections import defaultdict

sections = "20 1 18 4 13 6 10 15 2 17 3 19 7 16 8 11 14 9 12 5".split()
targets = set(r + s for r in 'SDT' for s in sections) | set(['SB', 'DB'])


def best_target(miss):
    "Return the target that maximizes the expected score."
    return max(targets, key=lambda t: expected_value(t, miss))


def expected_value(target, miss):
    "The expected score of aiming at target with a given miss ratio."
    return sum(value(t) * p for (t, p) in outcome(target, miss).items())


def outcome(target, miss):
    "Return a probability distribution of {target: probability} pairs."
    results = defaultdict(float)
    for (ring, ringP) in ring_outcome(target, miss):
        for (sect, sectP) in section_outcome(target, miss):
            if ring == 'S' and sect.endswith('B'):
                # Bull section, but ring missed out to the S ring:
                # spread the result over all 20 sections.
                for s in sections:
                    results[Target(ring, s)] += (ringP * sectP) / 20.
            else:
                results[Target(ring, sect)] += (ringP * sectP)
    return dict(results)


def ring_outcome(target, miss):
    "Return a probability distribution of [(ring, probability)] pairs."
    hit = 1.0 - miss
    r = target[0]
    if target == 'DB':  # misses tripled; can miss to SB or to S
        miss = min(3 * miss, 1.)
        hit = 1. - miss
        return [('DB', hit), ('SB', miss / 3.), ('S', 2. / 3. * miss)]
    elif target == 'SB':  # Bull can miss in either S or DB direction
        return [('SB', hit), ('DB', miss / 4.), ('S', 3 / 4. * miss)]
    elif r == 'S':  # miss ratio cut to miss/5
        return [(r, 1.0 - miss / 5.), ('D', miss / 10.), ('T', miss / 10.)]
    elif r == 'D':  # Double can miss either on board or off
        return [(r, hit), ('S', miss / 2), ('OFF', miss / 2)]
    elif r == 'T':  # Triple can miss in either direction, but both are S
        return [(r, hit), ('S', miss)]


def section_outcome(target, miss):
    "Return a probability distribution of [(section, probability)] pairs."
    hit = 1.0 - miss
    if target in ('SB', 'DB'):
        misses = [(s, miss / 20.) for s in sections]
    else:
        i = sections.index(target[1:])
        misses = [(sections[i - 1], miss / 2), (sections[(i + 1) % 20], miss / 2)]
    return [(target[1:], hit)] + misses


def Target(ring, section):
    "Construct a target name from a ring and section."
    if ring == 'OFF':
        return 'OFF'
    elif ring in ('SB', 'DB'):
        return ring if (section == 'B') else ('S' + section)
    else:
        return ring + section


def same_outcome(dict1, dict2):
    "Two outcome dicts are the same if all corresponding probabilities match."
    return all(abs(dict1.get(key, 0) - dict2.get(key, 0)) <= 0.0001
               for key in set(dict1) | set(dict2))


def test_darts2():
    assert best_target(0.0) == 'T20'
    assert best_target(0.1) == 'T20'
    assert best_target(0.4) == 'T19'
    assert same_outcome(outcome('T20', 0.0), {'T20': 1.0})
    assert same_outcome(outcome('T20', 0.1),
                        {'T20': 0.81, 'S1': 0.005, 'T5': 0.045,
                         'S5': 0.005, 'T1': 0.045, 'S20': 0.09})
    assert same_outcome(
        outcome('SB', 0.2),
        {'S9': 0.016, 'S8': 0.016, 'S3': 0.016, 'S2': 0.016, 'S1': 0.016,
         'DB': 0.04, 'S6': 0.016, 'S5': 0.016, 'S4': 0.016, 'S20': 0.016,
         'S19': 0.016, 'S18': 0.016, 'S13': 0.016, 'S12': 0.016, 'S11': 0.016,
         'S10': 0.016, 'S17': 0.016, 'S16': 0.016, 'S15': 0.016, 'S14': 0.016,
         'S7': 0.016, 'SB': 0.64})
    assert same_outcome(outcome('T20', 0.3),
                        {'S1': 0.045, 'T5': 0.105, 'S5': 0.045,
                         'T1': 0.105, 'S20': 0.21, 'T20': 0.49})
    assert best_target(0.6) == 'T7'
    return 'test_darts2 passes'


print(test_darts2())
print('best_target(0.4) =', best_target(0.4))


test_darts2 passes
best_target(0.4) = T19


## Unit 6 — Fun with Words (Portmanteau)

**Problem.** A portmanteau blends two words, like *math* + *athlete* = *mathlete*. Write `natalie(words)` returning the **best** portmanteau, or `None`. A valid portmanteau splits into three non-empty pieces `start + mid + end`, where both `start+mid` and `mid+end` are words in the list (and the two source words are different).

**Scoring.** A longer word is better; it is well-balanced when `mid` is about half the total length and `start`/`end` are about a quarter each. For total length `T = S+M+E`, the ideal lengths are `T/4, T/2, T/4`, so:

$$\text{score} = T - \left|S - \tfrac{T}{4}\right| - \left|M - \tfrac{T}{2}\right| - \left|E - \tfrac{T}{4}\right|$$

(e.g. `'adole'+'scent'+'ed'` → `12 - |5-3| - |5-6| - |2-3| = 8`).

**Key idea — two O(N) passes instead of O(N²).**
- **Pass 1 (`compute_ends`).** For every word, split it every way into `mid + end` and record a dict `{mid: [end, ...]}` (like Scrabble prefixes).
- **Pass 2 (`alltriples`).** For every word, split it into `start + mid` and look up the recorded endings for that `mid`. Each `(start, mid, end)` with `start+mid ≠ mid+end` (different words) is a candidate triple.
- Pick the triple with the highest `portman_score` and join it. This avoids the naive all-pairs O(N²) comparison.


from collections import defaultdict


def natalie(words):
    "Find the best Portmanteau word formed from any two of the list of words."
    # First find all (start, mid, end) triples, then find the best scoring one.
    triples = alltriples(words)
    if not triples:
        return None
    return ''.join(max(triples, key=portman_score))


def alltriples(words):
    """All (start, mid, end) triples where start+mid and mid+end are in words
    (and all three parts are non-empty)."""
    # Compute all {mid: [end]} pairs, then for each (start, mid) pair grab the
    # [end...] entries. Two O(N) passes, much better than the naive O(N^2).
    ends = compute_ends(words)
    return [(start, mid, end)
            for w in words
            for start, mid in splits(w)
            for end in ends[mid]
            if w != mid + end]


def splits(w):
    "Return a list of splits of the word w into two non-empty pieces."
    return [(w[:i], w[i:]) for i in range(1, len(w))]


def compute_ends(words):
    "Return a dict of {mid: [end, ...]} entries."
    ends = defaultdict(list)
    for w in words:
        for mid, end in splits(w):
            ends[mid].append(end)
    return ends


def portman_score(triple):
    "Return the numeric score for a (start, mid, end) triple."
    S, M, E = map(len, triple)
    T = S + M + E
    return T - abs(S - T / 4.) - abs(M - T / 2.) - abs(E - T / 4.)


In [17]:
def test_natalie():
    "Some test cases for natalie"
    assert (natalie(['eskimo', 'escort', 'kimchee', 'kimono', 'cheese'])
            == 'eskimono')
    assert (natalie(['kimono', 'kimchee', 'cheese', 'serious', 'us', 'usage'])
            == 'kimcheese')
    assert (natalie(['circus', 'elephant', 'lion', 'opera', 'phantom'])
            == 'elephantom')
    assert (natalie(['adolescent', 'scented', 'centennial', 'always',
                     'ado', 'centipede'])
            in ('adolescented', 'adolescentennial', 'adolescentipede'))
    assert (natalie(['programmer', 'coder', 'partying', 'merrymaking'])
            == 'programmerrymaking')
    assert (natalie(['int', 'intimate', 'hinter', 'hint', 'winter'])
            == 'hintimate')
    assert (natalie(['morass', 'moral', 'assassination'])
            == 'morassassination')
    assert (natalie(['entrepreneur', 'academic', 'doctor',
                     'neuropsychologist', 'neurotoxin', 'scientist', 'gist'])
            in ('entrepreneuropsychologist', 'entrepreneurotoxin'))
    assert (natalie(['perspicacity', 'cityslicker', 'capability', 'capable'])
            == 'perspicacityslicker')
    assert (natalie(['backfire', 'fireproof', 'backflow', 'flowchart',
                     'background', 'groundhog'])
            == 'backgroundhog')
    assert (natalie(['streaker', 'nudist', 'hippie', 'protestor',
                     'disturbance', 'cops'])
            == 'nudisturbance')
    assert (natalie(['night', 'day']) is None)
    assert (natalie(['dog', 'dogs']) is None)
    assert (natalie(['test']) is None)
    assert (natalie(['']) is None)
    assert (natalie(['ABC', '123']) is None)
    assert (natalie([]) is None)
    assert (natalie(['pedestrian', 'pedigree', 'green', 'greenery'])
            == 'pedigreenery')
    assert (natalie(['armageddon', 'pharma', 'karma', 'donald', 'donut'])
            == 'pharmageddon')
    assert (natalie(['lagniappe', 'appendectomy', 'append', 'lapin'])
            == 'lagniappendectomy')
    assert (natalie(['angler', 'fisherman', 'boomerang', 'frisbee', 'rangler',
                     'ranger', 'rangefinder'])
            in ('boomerangler', 'boomerangefinder'))
    assert (natalie(['freud', 'raelian', 'dianetics', 'jonestown', 'moonies'])
            == 'freudianetics')
    assert (natalie(['atheist', 'math', 'athlete', 'psychopath'])
            in ('psychopatheist', 'psychopathlete'))
    assert (natalie(['hippo', 'hippodrome', 'potato', 'dromedary'])
            == 'hippodromedary')
    assert (natalie(['taxi', 'taxicab', 'cabinet', 'cabin',
                     'cabriolet', 'axe'])
            in ('taxicabinet', 'taxicabriolet'))
    assert (natalie(['pocketbook', 'bookmark', 'bookkeeper', 'goalkeeper'])
            in ('pocketbookmark', 'pocketbookkeeper'))
    assert (natalie(['athlete', 'psychopath', 'athletic', 'axmurderer'])
            in ('psychopathlete', 'psychopathletic'))
    assert (natalie(['info', 'foibles', 'follicles'])
            == 'infoibles')
    assert (natalie(['moribund', 'bundlers', 'bundt'])
            == 'moribundlers')
    return 'test_natalie passes'


print(test_natalie())


test_natalie passes
